### Module 3 : Embeddings

#### Objective

Convert text chunks into dense numerical vectors that preserve semantic meaning.

These vectors will later be indexed inside a Vector Database (FAISS) for semantic retrieval.

---

Learning Outcomes

- Understand embeddings
- Learn semantic representation
- Generate embeddings
- Inspect embedding dimensions
- Prepare data for vector indexing

In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [17]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
print(model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [19]:
sentence = "Liquidity Coverage Ratio improves banking stability."

embedding = model.encode(sentence)
print("Embedding shape:", embedding.shape)

Embedding shape: (384,)


In [20]:
type(embedding)

numpy.ndarray

In [21]:
embedding[:10]

array([ 4.2341538e-02, -2.6959032e-02, -5.2073367e-02, -2.9219296e-02,
        6.7890227e-02,  1.0090441e-02, -2.6970034e-05,  2.6090541e-03,
       -4.6016656e-02, -5.2135326e-02], dtype=float32)

In [25]:
sentence1 = "The bank has sufficient liquidity."

sentence2 = "The financial institution has enough liquid assets."

emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
similarity = cosine_similarity(
    [emb1],
    [emb2]
)

print(similarity)

[[0.7748589]]


### Document Embedding

In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\shrab\AppData\Local\Temp\ipykernel_24960\2300941456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [27]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [29]:
pdf_path = r"../data/raw/basel/bcbs189.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()
type(documents), len(documents)

(list, 77)

In [30]:
from src.text_splitter import split_documents

chunks = split_documents(documents)

type(chunks), len(chunks)

(list, 305)

In [31]:
chunk_texts = [chunk.page_content for chunk in chunks]

In [32]:
print(len(chunk_texts))

305


In [33]:
print(chunk_texts[0])

Basel Committee 
on Banking Supervision  
Basel III: A global 
regulatory framework for 
more resilient banks and 
banking systems 
December 2010 
(rev June 2011) 
This standard has been integrated into the consolidated Basel Framework: https://www.bis.org/basel_framework/


In [34]:
# Generate embeddings for the chunks

embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

type(embeddings), embeddings.shape

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

(numpy.ndarray, (305, 384))